# L9 demo: a baseline ladder, three leaks, and one test set

We take the UCI **Combined Cycle Power Plant** data from raw rows to a defended
model choice, and along the way we produce three data-leakage bugs on purpose
and measure what each one costs.

The order matters. The bug the textbooks warn about turns out to be worth
almost nothing on this dataset, and the one nobody mentions is worth half the
reported accuracy. Watch which is which.

**The rule this notebook obeys:** the test set is split off in cell 4 and is not
looked at again until the final section. Everything in between uses
cross-validation on the training portion only.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


## Get the data

Two datasets, both from the UCI Machine Learning Repository, both small enough
to run on a laptop.

- **Combined Cycle Power Plant** (`Folds5x2_pp.xlsx`): 9,568 hourly records from
  one plant at full load, 2006-2011. Four ambient measurements predict net
  electrical output in MW. The primary dataset for this session.
- **Airfoil Self-Noise** (`airfoil_self_noise.dat`): 1,503 anechoic wind-tunnel
  measurements from NASA RP-1218. We use it for the grouped-data contrast,
  because its rows come in wind-tunnel runs and the power plant's do not.

The cell below downloads and caches both on first run. If UCI has moved the
files, the landing pages are
[dataset 294](https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant)
and [dataset 291](https://archive.ics.uci.edu/dataset/291/airfoil+self+noise);
drop `Folds5x2_pp.xlsx` and `airfoil_self_noise.dat` into `.cache/` by hand and
everything below works unchanged.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)

SOURCES = {
    'Folds5x2_pp.xlsx': (
        'https://archive.ics.uci.edu/static/public/294/combined+cycle+power+plant.zip',
        'CCPP/Folds5x2_pp.xlsx'),
    'airfoil_self_noise.dat': (
        'https://archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip',
        'airfoil_self_noise.dat'),
}

for local, (url, member) in SOURCES.items():
    if (CACHE / local).exists():
        continue
    print('downloading', url)
    with urllib.request.urlopen(url) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    (CACHE / local).write_bytes(archive.read(member))

print('cached:', sorted(p.name for p in CACHE.iterdir()))

## Stage 1: the data card, and one thing worth checking

Before modelling anything, write down what the columns are and what the rows
are. Units first, because a column of floats does not carry them.

| column | meaning | units |
|---|---|---|
| `AT` | ambient temperature | °C |
| `V`  | exhaust vacuum | cm Hg |
| `AP` | ambient pressure | millibar |
| `RH` | relative humidity | % |
| `PE` | **target**: net hourly electrical output | MW |

Then look at the file itself. It has five sheets, and the readme says why:

> For comparability with our baseline studies, and to allow 5x2 fold
> statistical tests be carried out, we provide the data shuffled five times.

Verify that claim rather than believing it.

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ['AT', 'V', 'AP', 'RH']
TARGET = 'PE'

book = pd.ExcelFile(CACHE / 'Folds5x2_pp.xlsx')
sheets = {name: pd.read_excel(book, name) for name in book.sheet_names}
ccpp = sheets['Sheet1']

# Are the five sheets five orderings of one table, or five different samples?
key = FEATURES + [TARGET]
canonical = ccpp.sort_values(key).reset_index(drop=True)
same_rows = {n: sheets[n].sort_values(key).reset_index(drop=True).equals(canonical)
             for n in book.sheet_names}
same_order = {n: sheets[n].equals(ccpp) for n in book.sheet_names}

print('sheets:', book.sheet_names)
print('same set of rows as Sheet1: ', same_rows)
print('same ORDER as Sheet1:       ', same_order)
print()
print(ccpp.describe().T[['count', 'mean', 'std', 'min', 'max']].round(2))

Five sheets, identical row sets, five different orderings. That is a defensible
choice for reproducing a specific 5x2 cross-validation protocol, and it has a
consequence nobody flags:

**Six years of hourly measurements arrived with their temporal order
destroyed.** You cannot plot the target against time, compute an
autocorrelation, or run a `TimeSeriesSplit`. Which means you cannot check the
exchangeability assumption that justifies the k-fold everybody runs on this
dataset, including this notebook.

And you have physical reason to doubt it. Ambient temperature at 2 p.m. and at
3 p.m. on the same day is nearly the same number, so consecutive hourly rows are
near-duplicates, and a random k-fold will systematically split those pairs
across the boundary.

We proceed with k-fold, because it is the best available choice given what was
published, and we write the caveat down rather than pretending it is not there.

While the five shuffles are here, they buy us one genuinely useful number: how
much a cross-validation estimate moves for reasons that have nothing to do with
the model. That is the yardstick every difference below has to clear.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

SEED = 0
RMSE = 'neg_root_mean_squared_error'

# The readme's own protocol: 2-fold CV on each of the five published shuffles.
noise_scores = np.concatenate([
    -cross_val_score(LinearRegression(), frame[FEATURES], frame[TARGET],
                     cv=KFold(2, shuffle=False), scoring=RMSE)
    for frame in sheets.values()
])

CV_NOISE = noise_scores.std()
print(f'5x2 CV of one linear model: {noise_scores.mean():.4f} MW, '
      f'std {CV_NOISE:.4f}, range {np.ptp(noise_scores):.4f}')
print(f'\nAnything smaller than about {2 * CV_NOISE:.2f} MW is a tie.')

## Stage 2: lock the test set, now, before looking at anything

This is the only cell that touches `train_test_split`, and `X_test` does not
appear again until the last section. Splitting first is a structural habit: it
is much harder to accidentally consult a test set you have not got a variable
pointing at.

In [ ]:
from sklearn.model_selection import train_test_split

X = ccpp[FEATURES].to_numpy()
y = ccpp[TARGET].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED)

print(f'train {X_train.shape[0]:,} rows   test {X_test.shape[0]:,} rows')
print('Everything below uses X_train only, via cross-validation.')

## Stage 3: the baseline ladder

Five models, from "predict a constant" upward, scored three ways in one pass.
The point of the ladder is that it gives every later number a unit: an RMSE of
3.4 MW means nothing until you know that doing nothing costs 17 and that a
straight line costs 4.6.

Note the second row. It is a linear regression on **ambient temperature alone**,
and it is here because a gas turbine breathes air: colder air is denser, so the
compressor ingests more mass, so the machine makes more power. That is the
domain baseline, and it is the one most studies skip.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor

cv = KFold(5, shuffle=True, random_state=SEED)
AT_ONLY = [FEATURES.index('AT')]

ladder = [
    ('predict the mean', DummyRegressor(strategy='mean'), None),
    ('linear, ambient temp only', LinearRegression(), AT_ONLY),
    ('linear, all four inputs', LinearRegression(), None),
    ('decision tree, depth 5', DecisionTreeRegressor(max_depth=5, random_state=SEED), None),
    ('random forest, 100 trees', RandomForestRegressor(100, random_state=SEED, n_jobs=-1), None),
]

rows = []
for name, estimator, columns in ladder:
    features = X_train if columns is None else X_train[:, columns]
    scores = cross_validate(
        estimator, features, y_train, cv=cv,
        scoring=('neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'))
    rows.append({
        'model': name,
        'RMSE': -scores['test_neg_root_mean_squared_error'].mean(),
        'fold std': scores['test_neg_root_mean_squared_error'].std(),
        'MAE': -scores['test_neg_mean_absolute_error'].mean(),
        'R2': scores['test_r2'].mean(),
    })

results = pd.DataFrame(rows).set_index('model')
floor, best = results['RMSE'].iloc[0], results['RMSE'].min()
results['gap closed'] = ((floor - results['RMSE']) / (floor - best)).map('{:.0%}'.format)
results.round(3)

Read the `gap closed` column, not the RMSE column.

Predicting the mean costs the target's own standard deviation, by construction.
**One thermodynamic variable and a straight line close about 85% of everything a
model can do here.** Every model past that is arguing over the last couple of
megawatts, and the fold standard deviation tells you which of those arguments
are real.

That framing is what you should put in a report. "The random forest achieves
3.5 MW" is a fact about nothing; the ladder tells a reader whether the last
increment justifies a scikit-learn dependency in a control room.

One deliberate discrepancy worth noticing: the figure in the notes reports the
same ladder at 17.07 / 5.43 / 3.36 MW, and this notebook reports roughly
17.06 / 5.46 / 3.49. The figure cross-validates on all 9,568 rows; this notebook
only ever sees the 7,654 in the training split, because the rest is locked away.
The constant and linear baselines barely move, and the random forest loses about
0.13 MW. That is the learning curve talking: the forest's validation error was
still falling at the right-hand edge of the plot, so taking a fifth of its
training data away costs it something real. The flexible model is the one that
notices.

## Leak 1: fit the scaler before splitting

The canonical warning. Standardise the whole feature matrix, then
cross-validate on the scaled version: every fold's held-out rows contributed to
the mean and standard deviation that transformed its training rows.

Predict the size of the effect before running the cell.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

leaked = StandardScaler().fit_transform(X_train)   # <-- fitted on every row

candidates = [
    ('LinearRegression', LinearRegression()),
    ('Ridge(alpha=100)', Ridge(100)),
    ('KNeighbors(k=5)', KNeighborsRegressor(5)),
    ('RandomForest', RandomForestRegressor(50, random_state=SEED, n_jobs=-1)),
]

for name, estimator in candidates:
    leaky = -cross_val_score(estimator, leaked, y_train, cv=cv, scoring=RMSE).mean()
    honest = -cross_val_score(make_pipeline(StandardScaler(), estimator),
                              X_train, y_train, cv=cv, scoring=RMSE).mean()
    print(f'{name:18s} leaky {leaky:7.4f}   honest {honest:7.4f}   '
          f'cost of the leak {honest - leaky:+.5f} MW')

print(f'\nFor reference, this dataset\'s fold-to-fold noise is +/-{CV_NOISE:.4f} MW.')

**The leak is real, and it is worth nothing.** The largest effect across four
models is a thousandth of a megawatt, and in some cases the leaky pipeline
scores very slightly *better*.

The reason is arithmetic rather than luck: a mean computed over 7,654 rows and a
mean computed over 6,123 randomly chosen ones are almost the same number. There
is no dramatic result available here, and inventing one would teach the wrong
lesson.

The right lesson is the uncomfortable one. **You cannot audit for leakage by
looking at your metrics.** This bug was invisible. The only reliable check is to
read the code, find every `.fit()` call, and ask what was in scope when it ran.
Wrap preprocessing in a `Pipeline` because it makes the bug structurally hard to
write, not because the score will thank you.

## Leak 2: choose the features before cross-validating

Same class of bug, five hundred times the consequence. This is the experiment
from section 7.10.2 of *The Elements of Statistical Learning*, and it is the
reason that section exists.

Generate data with **no signal at all**: 50 samples, 5,000 predictors of pure
Gaussian noise, and labels assigned by coin flip. The true error rate of any
classifier on this problem is 50%.

Now screen the 5,000 columns down to the 100 most associated with the label, and
cross-validate a classifier on those 100 columns.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

N_SAMPLES, N_FEATURES, N_KEEP, N_REPEATS = 50, 5000, 100, 20

outside, inside = [], []
for repeat in range(N_REPEATS):
    rng = np.random.default_rng(repeat)
    noise = rng.standard_normal((N_SAMPLES, N_FEATURES))
    coin = rng.integers(0, 2, N_SAMPLES)
    folds = StratifiedKFold(5, shuffle=True, random_state=repeat)

    # WRONG: screening sees every row, including the ones it will be scored on.
    screened = SelectKBest(f_classif, k=N_KEEP).fit_transform(noise, coin)
    outside.append(1 - cross_val_score(KNeighborsClassifier(1), screened, coin,
                                       cv=folds).mean())

    # RIGHT: the identical screening step, refitted inside every training fold.
    guarded = Pipeline([('screen', SelectKBest(f_classif, k=N_KEEP)),
                        ('model', KNeighborsClassifier(1))])
    inside.append(1 - cross_val_score(guarded, noise, coin, cv=folds).mean())

print(f'screen first, then cross-validate : {np.mean(outside):6.1%} error')
print(f'screen inside every training fold : {np.mean(inside):6.1%} error')
print(f'the truth                         : {50.0:6.1f}% error')

A classifier of nothing, reported as nearly perfect.

With 5,000 candidates and 50 observations, some columns correlate with the
labels by chance. Selecting them using the labels of rows you are about to
validate against hands the classifier the answer.

This is Google Flu Trends in miniature: 50 million candidate search queries
fitted to 1,152 observations, and a model that turned out to be, in the
post-mortem's phrase, "part flu detector, part winter detector." The number of
candidates does not have to be enormous for this to bite. It has to be large
relative to your sample size, and in a wide sensor matrix it usually is.

## Leak 3: the split itself

The two above happen inside a modelling pipeline, so a `Pipeline` can defend
against them. This one happens **before** any pipeline sees the data, which is
why no amount of scikit-learn hygiene will catch it.

The power-plant rows are close to independent, so switch datasets. Each airfoil
row is one *frequency* within one wind-tunnel configuration, and a configuration
is a choice of chord length, free-stream velocity, angle of attack and
displacement thickness. Group by configuration and 1,503 rows collapse into
about a hundred groups.

At deployment you will be asked about a configuration nobody ran. A random
k-fold never asks that question.

In [ ]:
from sklearn.model_selection import GroupKFold

AIRFOIL_FEATURES = ['freq_hz', 'aoa_deg', 'chord_m', 'velocity_ms', 'thickness_m']
airfoil = pd.read_csv(CACHE / 'airfoil_self_noise.dat', sep='\t', header=None,
                      names=AIRFOIL_FEATURES + ['spl_db'])

# A group is a tunnel configuration. Frequency is swept WITHIN a run, so it is a
# feature, not part of the group key.
groups = airfoil.groupby(['aoa_deg', 'chord_m', 'velocity_ms',
                          'thickness_m']).ngroup().to_numpy()
Xa = airfoil[AIRFOIL_FEATURES].to_numpy()
ya = airfoil['spl_db'].to_numpy()

print(f'{len(airfoil):,} rows in {groups.max() + 1} configurations, '
      f'median {int(pd.Series(groups).value_counts().median())} rows each\n')

for name, estimator in [
        ('predict the mean', DummyRegressor(strategy='mean')),
        ('linear', LinearRegression()),
        ('k-NN, k=5', make_pipeline(StandardScaler(), KNeighborsRegressor(5))),
        ('random forest', RandomForestRegressor(200, random_state=SEED, n_jobs=-1))]:
    random_folds = -cross_val_score(estimator, Xa, ya, scoring=RMSE,
                                    cv=KFold(5, shuffle=True, random_state=SEED))
    whole_runs = -cross_val_score(estimator, Xa, ya, scoring=RMSE,
                                  cv=GroupKFold(5), groups=groups)
    print(f'{name:18s} KFold {random_folds.mean():6.3f} dB   '
          f'GroupKFold {whole_runs.mean():6.3f} dB   '
          f'inflation {whole_runs.mean() / random_folds.mean():.2f}x')

The random forest reports 1.76 dB under a random k-fold and 2.69 dB when whole
runs are held out. Same data, same model, same code, and the number a paper
would print is **52% too low**.

Two details are worth more than the headline.

**Which models the leak reaches.** The mean and linear baselines report the same
number either way, because neither has the capacity to memorise one run's curve.
k-NN, which is nothing but memorisation, inflates by 28%. The forest inflates
most. A gap of zero is never evidence that your split was sound; it may only be
evidence that your model was too rigid to exploit it.

**What did not change.** The ranking survived: the forest is still best and the
linear model still worse. What broke was the number, not the decision. That will
not always be true, and it is a poor thing to rely on.

## The fix, and a defensible selection

Now do it properly on the power-plant training data: every transform inside a
`Pipeline`, a grid searched with cross-validation, and the winner chosen by the
**one-standard-error rule** rather than by raw minimum.

The 1-SE rule says: find the best validation score, compute the standard error
of that estimate, and among all candidates whose score falls within one standard
error of the best, take the **simplest**. The ranking inside that band is noise,
and simplicity is not.

"Simplest" is not a property scikit-learn knows about, so we declare it: a
`complexity` rank over the three families, lowest first. Making that ordering
explicit is the whole point. A selection rule you cannot write down is a
preference, not a rule.

With 7,654 training rows and 20 candidates, the selection-bias measurement from
the notes says the winner's curse here is worth a few thousandths of a megawatt,
so a single cross-validation loop is enough and nested CV would be a waste. On a
200-specimen dataset it would not be.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures


def ridge_poly(degree, alpha):
    return Pipeline([
        ('poly', PolynomialFeatures(degree, include_bias=False)),
        ('scale', StandardScaler()),
        ('model', Ridge(alpha=alpha)),
    ])


# Three model families, each listed simplest-first, and the `complexity` rank
# is an explicit statement of what "simpler" means here: fewer moving parts to
# explain, to audit, and to keep running in a control room. Within a family, a
# stronger penalty, a larger k, and fewer/shallower trees all count as simpler.
CANDIDATES = (
    [(f'ridge, linear terms, alpha={a:g}', 1 + i / 10, ridge_poly(1, a))
     for i, a in enumerate(np.logspace(3, -2, 6))]
    + [(f'ridge, quadratic terms, alpha={a:g}', 2 + i / 10, ridge_poly(2, a))
       for i, a in enumerate(np.logspace(3, -2, 6))]
    + [(f'k-NN, k={k}', 3 + i / 10,
        make_pipeline(StandardScaler(), KNeighborsRegressor(k)))
       for i, k in enumerate((20, 10, 5, 3))]
    + [(f'random forest, {n} trees, depth {d}', 4 + i / 10,
        RandomForestRegressor(n, max_depth=d, random_state=SEED, n_jobs=-1))
       for i, (n, d) in enumerate([(100, 6), (100, 12), (100, None), (300, None)])]
)

study = pd.DataFrame([
    dict(model=name, complexity=rank, rmse=fold.mean(),
         se=fold.std() / np.sqrt(cv.get_n_splits()), estimator=estimator)
    for name, rank, estimator in CANDIDATES
    for fold in [-cross_val_score(estimator, X_train, y_train, cv=cv,
                                  scoring=RMSE, n_jobs=-1)]
])

best = study.loc[study['rmse'].idxmin()]
threshold = best['rmse'] + best['se']
within = study[study['rmse'] <= threshold]
chosen = within.sort_values(['complexity', 'rmse']).iloc[0]

print(f'{len(study)} candidates over 3 model families')
print()
print(study.sort_values('rmse')[['model', 'complexity', 'rmse', 'se']]
      .head(8).to_string(index=False, float_format='%.4f'))
print()
print(f'best validation RMSE  {best["rmse"]:.4f}   ({best["model"]})')
print(f'1-SE threshold        {threshold:.4f}   -> {len(within)} candidates qualify')
print(f'chosen by the rule    {chosen["rmse"]:.4f}   ({chosen["model"]})')
print(f'the rule gave up      {chosen["rmse"] - best["rmse"]:.4f} MW, '
      f'{(chosen["rmse"] - best["rmse"]) / CV_NOISE:.2f}x the fold noise')

The rule did something small and real: the best raw score belongs to a
300-tree forest, but a 100-tree forest is inside one standard error of it, so we
ship the one that costs a third as much to fit. The difference we gave up is a
small fraction of this dataset's own fold-to-fold noise, which is another way of
saying we gave up nothing we could have measured.

That is what the rule is for. It stops you paying real money for a ranking that
is noise.

### Error analysis, before the test set

No scalar metric tells you that your errors are *structured*, and structured
errors are where the physics is hiding. Look at the residuals from
cross-validated predictions, which are honest in the sense that every prediction
was made by a model that had not seen that row.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict

final = chosen['estimator']
oof = cross_val_predict(final, X_train, y_train, cv=cv)
residuals = y_train - oof

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(oof, residuals, s=4, alpha=0.20, color='#1f5c99')
axes[0].axhline(0, color='#c41230', lw=1.2)
axes[0].set_xlabel('predicted output, MW')
axes[0].set_ylabel('residual, MW')
axes[0].set_title('Residuals vs fitted')

axes[1].scatter(X_train[:, FEATURES.index('AT')], residuals, s=4, alpha=0.20,
                color='#2b7a4b')
axes[1].axhline(0, color='#c41230', lw=1.2)
axes[1].set_xlabel('ambient temperature, °C')
axes[1].set_ylabel('residual, MW')
axes[1].set_title('Residuals vs the dominant input')
for ax in axes:
    ax.grid(True, color='#d8d8d8', lw=0.7)
    ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

print(f'residual mean {residuals.mean():+.4f} MW, std {residuals.std():.4f} MW')
print(f'|residual| > 3 std: {(np.abs(residuals) > 3 * residuals.std()).sum()} rows')

Look for curvature, for a fan shape, and for bias at the extremes of the range.
A model with a good RMSE and a smile-shaped residual plot has a correctable
deficiency that no summary statistic would have shown you.

## Touch the test set. Once.

Everything up to here used `X_train`. Now fit the chosen configuration on all of
it and evaluate once, with an uncertainty attached, because a test estimate is
one number with sampling error and reporting it to four figures is a claim you
did not earn.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

final.fit(X_train, y_train)
predictions = final.predict(X_test)

test_rmse = float(np.sqrt(np.mean((y_test - predictions) ** 2)))

# Bootstrap the test set to put an interval on that number.
rng = np.random.default_rng(SEED)
boot = np.array([
    np.sqrt(np.mean((y_test[idx] - predictions[idx]) ** 2))
    for idx in (rng.integers(0, len(y_test), len(y_test)) for _ in range(2000))
])
low, high = np.percentile(boot, [2.5, 97.5])

print(f'FINAL, on {len(y_test):,} rows never used for any decision:')
print(f'  RMSE  {test_rmse:.3f} MW   95% bootstrap interval [{low:.3f}, {high:.3f}]')
print(f'  MAE   {mean_absolute_error(y_test, predictions):.3f} MW')
print(f'  R2    {r2_score(y_test, predictions):.4f}')
print(f'\nfor comparison, predicting the mean costs {y_test.std():.3f} MW')

## What to take away

**Report the ladder, not the number.** The final RMSE above is meaningful only
next to the 17 MW that predicting a constant costs and the 5.4 MW that one
thermodynamic variable and a straight line cost.

**The size of a leak tells you about that leak, not about leakage.** Scaling
before the split cost 0.001 MW here and the group split cost 52% on the airfoil
data. Neither number generalises. Audit the code.

**Attach an interval to the test number, and say how the model was selected.**
"3.4 MW" is an assertion. "3.4 MW, 95% bootstrap interval [3.2, 3.6], chosen by
a 1-SE rule over 21 candidates on 5-fold CV of a locked 80% training split" is a
result.

**And write down what you could not check.** This dataset arrived shuffled, so
the exchangeability assumption behind every k-fold in this notebook is
unverifiable. That belongs in the report.

Next session takes this study and makes it auditable: every run above logged to
MLflow with its parameters, metrics, dataset hash and git commit, plus a
systematic hyperparameter search with Optuna.